# Assessment · Module 02 · Data literacy and EDA

**Cumulative, across chapters 02-01 to 02-08.** Allow about **75 minutes**. Work it **without notes
and without rereading the chapters** - that is the whole point. Looking things up turns an assessment
into a reading exercise and tells you nothing about what you actually retained.

**40 marks in three parts:**

| Part | What it tests | Marks |
|---|---|---|
| **A** | Recall - ten short questions | 10 |
| **B** | Doing - eight quantities computed from a dataset, self-checking | 16 |
| **C** | Judgement - three questions with no single right answer | 14 |

Part B marks itself. Parts A and C are marked against
`assessments/module_02_assessment_solutions.ipynb` **after** you have written your answers down.

At the end there is a score table and a **remediation map** - which chapter to revisit for each
question you missed. A low score here is worth more than a high score you got by looking things up.

---

## The data

One row per **loan event** at a city library with two branches, for one quarter. It is synthetic, it
was generated with a fixed seed, and it has been given the kinds of defect this module is about.
Nothing has been made obvious.

| Column | Meaning |
|---|---|
| `loan_id` | identifier for the loan |
| `member_id` | identifier for the borrower |
| `branch` | `central` or `riverside` |
| `format` | `print` or `ebook` |
| `days_kept` | days between borrowing and return |
| `renewed` | 1 if the loan was renewed at least once, 0 otherwise |

Run the next cell, then start at Part A.

In [ ]:
import hashlib

import numpy as np
import pandas as pd


def load_loans():
    """SYNTHETIC. One row per loan event at a two-branch city library."""
    rng = np.random.default_rng(2026)
    n_members = 120
    intensity = rng.gamma(2.0, 1.0, n_members) + 0.3
    loans_each = np.maximum(1, np.round(intensity * 4)).astype(int)
    member_id = np.repeat(np.arange(1, n_members + 1), loans_each)
    n = len(member_id)
    mu = np.clip(34 - 7.0 * intensity[member_id - 1], 5, None)
    days = np.clip(np.round(rng.gamma(4.0, mu / 4.0)), 1, None)
    days = np.minimum(days, 28)
    branch = rng.choice(["central", "riverside"], n, p=[0.45, 0.55])
    fmt = np.where(rng.random(n) < np.where(branch == "central", 0.75, 0.25), "print", "ebook")
    base = np.where(branch == "central", 0.62, 0.38)
    renewed = (rng.random(n) < base + np.where(fmt == "ebook", 0.10, 0.0)).astype(int)
    days = np.where(rng.random(n) < 0.08, -1, days)
    loans = pd.DataFrame({"loan_id": np.arange(1, n + 1), "member_id": member_id,
                          "branch": branch, "format": fmt,
                          "days_kept": days.astype(int), "renewed": renewed})
    return pd.concat([loans, loans.iloc[100:175].copy()], ignore_index=True)


EXPECTED = {
    "B1": "3b90cedb8dc1", "B2": "b3ba5ed83382", "B3": "3f408706c437", "B4": "13d621feea07",
    "B5": "c1a2fb15ff02", "B6": "b77f7f05789d", "B7": "333e2cde6851", "B8": "9563004c0f78",
}


def check(task, answer):
    """Marks one Part B answer without revealing it. Counts: whole numbers. Rates and means: 2 dp."""
    task = task.upper()
    if task not in EXPECTED:
        print("unknown task:", task)
        return
    candidates = [answer] if isinstance(answer, (int, np.integer)) else [
        round(float(answer) + delta, 2) for delta in (-0.01, 0.0, 0.01)
    ]
    for value in candidates:
        text = str(int(value)) if isinstance(answer, (int, np.integer)) else "%.2f" % value
        if hashlib.sha256((task + "|" + text).encode()).hexdigest()[:12] == EXPECTED[task]:
            print("%s  correct" % task)
            return
    print("%s  not yet - check your working, then try again" % task)


loans = load_loans()
print("loaded %d rows, %d columns" % loans.shape)
print(loans.head(3).to_string(index=False))

## Part A · Recall (10 marks, 1 each)

Answer from memory in the markdown cell below. One or two sentences each. Do not run any code for
this part.

**A1.** What does one row represent in the loans table, and name one quantity that a per-row average
would get wrong.

**A2.** You find that 6% of rows are exact duplicates. Give one plausible mechanism, and say whether
dropping them is always the right fix.

**A3.** A column stores `-1` where the value is unknown. Name two ways this is more dangerous than
storing a genuine missing value.

**A4.** What is the difference between missing-at-random and missing-not-at-random, and which one
does mean-imputation make worse?

**A5.** How do you tell a ceiling from a genuine maximum, using only the data?

**A6.** Why can a three-sigma outlier rule fail to flag an outlier that is obvious to the eye?

**A7.** State Simpson's paradox in one sentence, and name the condition on the data that produces it.

**A8.** A subgroup answer and a pooled answer disagree. What kind of question decides which one to
use, and what are the two cases?

**A9.** You screen 30 columns against a target on 100 rows and the best correlation is 0.28. What
must you compare it against before reporting it?

**A10.** Name two questions exploratory analysis can settle and two it cannot.

### Your Part A answers

*Write here. Do not scroll to the solutions until all ten are written.*

**A1.**

**A2.**

**A3.**

**A4.**

**A5.**

**A6.**

**A7.**

**A8.**

**A9.**

**A10.**

## Part B · Doing (16 marks, 2 each)

Compute each quantity from `loans`. Check yourself with `check("B1", your_answer)`.

**Counts are whole numbers. Rates, percentages and means are to 2 decimal places.**

The tasks are in a deliberate order, and getting a later one right depends on having handled an
earlier one. If a check fails, the error is at least as likely to be upstream as in the line you
just wrote.

**B1.** How many rows are exact duplicates of another row?

**B2.** By what percentage does the delivered row count overstate the number of distinct loans?
(A count of 110 where the truth is 100 overstates by 10.00.)

**B3.** Working from the de-duplicated data: how many loans have not been returned?

**B4.** The mean of `days_kept` on the de-duplicated data, computed naively - no handling of anything.

**B5.** The mean number of days a *returned* loan was kept.

**B6.** How many returned loans sit exactly at the largest value of `days_kept`?

**B7.** The mean, across members, of each member's own mean `days_kept` (returned loans only).

**B8.** The overall renewal rate for `ebook` loans, on the de-duplicated data.

In [ ]:
# Part B workspace. Use check("B1", answer) as you go.

## Part C · Judgement (14 marks)

No single right answer. The solutions give one defensible response and say what makes it defensible.

**C1 (5 marks).** The library director wants to know **which format gets renewed more**, to decide
what to buy next year.

- Compute the renewal rate by format overall, and by format within each branch.
- The two answers disagree. Say which one you would give the director and **why**.
- State what would have to be true for the other answer to be the right one.

**C2 (4 marks).** B4 and B5 differ, and B5 and B7 differ. Explain what each difference is caused by,
and which of the three numbers you would put in a report titled *"How long do our members keep
books?"* - defending the choice.

**C3 (5 marks).** Write the four-line honest finding for the claim *"ebooks are renewed more often
than print books"*: the claim in the weakest verb that is still true, the evidence with its numbers,
what else could produce it, and what would change your mind. Then say, in one sentence, whether this
dataset can tell the director what would happen if the library bought more ebooks.

In [ ]:
# Part C workspace.

### Your Part C answers

**C1.**

**C2.**

**C3.**

## Scoring

Mark Parts A and C against the solutions notebook. Part B marked itself: 2 marks per `correct`.

| Part | Available | Yours |
|---|---|---|
| A | 10 | |
| B | 16 | |
| C | 14 | |
| **Total** | **40** | |

**What your score means:**

| Score | Reading |
|---|---|
| 34-40 | Module 02 is in place. Go to 03-01 |
| 26-33 | Solid. Revisit the specific chapters below for what you missed, then go on |
| 18-25 | The ideas are there and the practice is not. Redo the exercises in the two or three chapters your misses cluster in |
| below 18 | Rework the module. It is eight chapters and it is the foundation for everything after - the cost of redoing it now is much lower than the cost of not having it in module 07 |

### Remediation map

| Missed | Revisit |
|---|---|
| A1, B7, C2 | **02-01** what is a row |
| A2, B1, B2 | **02-02** provenance, and **02-04** duplicates |
| A3, A4, B3, B4 | **02-04** missing values and sentinels |
| A5, A6, B5, B6 | **02-05** distributions, outliers, ceilings |
| A7, A8, C1 | **02-06** looking at two things at once |
| A9, A10, C3 | **02-07** honest reporting and the limits of EDA |
| Several parts of B in sequence | **02-08** the applied workflow and its order |

A useful thing to notice: if your misses cluster in one row of that table, you have one gap and it is
cheap to close. If they are spread evenly, the problem is usually pace rather than any one topic.

---

**Next:** module 03 begins the mathematics - the smallest amount that makes everything afterwards
readable, introduced where it is used. Start with **03-01**.